In [1]:
place_to_full_name = {
    "Adams": "Adams city, Tennessee",
    "Ashland City": "Ashland City town, Tennessee",
    "Belle Meade": "Belle Meade city, Tennessee",
    "Berry Hill": "Berry Hill city, Tennessee",
    "Brentwood": "Brentwood city, Tennessee",
    "Burns": "Burns town, Tennessee",
    "Cedar Hill": "Cedar Hill city, Tennessee",
    "Charlotte": "Charlotte town, Tennessee",
    "Clarksville": "Clarksville city, Tennessee",
    "Columbia": "Columbia city, Tennessee",
    "Coopertown": "Coopertown town, Tennessee",
    "Cross Plains": "Cross Plains city, Tennessee",
    "Cumberland City": "Cumberland City town, Tennessee",
    "City of Dickson": "Dickson city, Tennessee",
    "Dover": "Dover city, Tennessee",
    "Eagleville": "Eagleville city, Tennessee",
    "Erin": "Erin city, Tennessee",
    "Fairview": "Fairview city, Tennessee",
    "Forest Hills": "Forest Hills city, Tennessee",
    "Franklin": "Franklin city, Tennessee",
    "Gallatin": "Gallatin city, Tennessee",
    "Goodlettsville": "Goodlettsville city, Tennessee",
    "Greenbrier": "Greenbrier town, Tennessee",
    "Trousdale/Hartsville": "Hartsville/Trousdale County, Tennessee",
    "Hendersonville": "Hendersonville city, Tennessee",
    "Kingston Springs": "Kingston Springs town, Tennessee",
    "LaVergne": "La Vergne city, Tennessee",
    "Lebanon": "Lebanon city, Tennessee",
    "McEwen": "McEwen city, Tennessee",
    "Millersville": "Millersville city, Tennessee",
    "Mitchellville": "Mitchellville city, Tennessee",
    "Mt. Juliet": "Mount Juliet city, Tennessee",
    "Mount Pleasant": "Mount Pleasant city, Tennessee",
    "Murfreesboro": "Murfreesboro city, Tennessee",
    "Metropolitan Nashville-Davidson County": "Nashville-Davidson metropolitan government (balance), Tennessee",
    "New Johnsonville": "New Johnsonville city, Tennessee",
    "Nolensville": "Nolensville town, Tennessee",
    "Oak Hill": "Oak Hill city, Tennessee",
    "Orlinda": "Orlinda city, Tennessee",
    "Pegram": "Pegram town, Tennessee",
    "Pleasant View": "Pleasant View city, Tennessee",
    "Portland": "Portland city, Tennessee",
    "Ridgetop": "Ridgetop city, Tennessee",
    "Slayden": "Slayden town, Tennessee",
    "Smyrna": "Smyrna town, Tennessee",
    "Spring Hill": "Spring Hill city, Tennessee",
    "Springfield": "Springfield city, Tennessee",
    "Tennessee Ridge": "Tennessee Ridge town, Tennessee",
    "Thompson's Station": "Thompson's Station town, Tennessee",
    "Vanleer": "Vanleer town, Tennessee",
    "Watertown": "Watertown city, Tennessee",
    "Waverly": "Waverly city, Tennessee",
    "Westmoreland": "Westmoreland town, Tennessee",
    "White Bluff": "White Bluff town, Tennessee",
    "White House": "White House city, Tennessee"
}

In [2]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import requests
pd.set_option('display.max_rows', 1000); pd.set_option('display.max_columns', 1000); pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

So I'm taking the sample download of parcel data from UrbanSim and using the parcel lookup Max had in his old spreadsheet to re-create the sheets in the same spreadsheet: SmallArea and the Summary_w_UGB. 

In [4]:
#we want 23 and 50
twentythree = pd.read_csv('../data/urbansim/Run38/Run 38 - 2023 - parcels v20250515.csv')
fifty = pd.read_csv('../data/urbansim/Run38/Run 38 - 2050 - parcels v20250515.csv')
dfs = [twentythree, fifty]
#easier not to concatenate yet
#data = pd.concat(dfs).reset_index(drop = True)
#data = data[['year', 'geo_level_id', 'household_population', 'sum_total_households', 'sum_total_jobs']]

urbansim_r is geo_level_id

In [5]:
#rename to id and make sure it's an integer for both dataframes
twentythree = twentythree.rename(columns = {'geo_level_id': 'ID'})
twentythree['ID'] = twentythree['ID'].astype(int)
fifty = fifty.rename(columns = {'geo_level_id': 'ID'})
fifty['ID'] = fifty['ID'].astype(int)

In [6]:
#read in the parcel crosswalk - big file takes a minute
parcel_cross_dl = pd.read_excel(r'G:\DATA\REQUESTS_CLIENTS\GNRC\20240215_UrbanSim_Update\Model Outputs\Small_Area.xlsx', sheet_name = 'parcel lookup')

In [7]:
parcel_cross = parcel_cross_dl.rename(columns = {'urbansim_r': 'ID'})

In [8]:
parcel_cross.head()

,OBJECTID *,Shape *,Join_Count,TARGET_FID,ID,X,Y,X_LP,Y_LP,NAME,NAMELSAD,Shape_Length,Name,Type
0,669347,Point,1,669346,15324,-87.013953,36.135439,-87.013953,36.135439,NaN,Cheatham,953969.243122,Peagram UGB,UGB
1,669350,Point,1,669349,15351,-87.011172,36.135244,-87.011172,36.135244,NaN,Cheatham,953969.243122,Peagram UGB,UGB
2,669352,Point,1,669351,15330,-87.013097,36.134054,-87.013097,36.134054,NaN,Cheatham,953969.243122,Peagram UGB,UGB
3,669353,Point,1,669352,15350,-87.009619,36.134734,-87.009619,36.134734,NaN,Cheatham,953969.243122,Peagram UGB,UGB
4,669354,Point,1,669353,15273,-87.012648,36.132675,-87.012648,36.132675,NaN,Cheatham,953969.243122,Peagram UGB,UGB


In [9]:
parcel_cross.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 886346 entries, 0 to 886345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   OBJECTID *    886346 non-null  int64  
 1   Shape *       886346 non-null  object 
 2   Join_Count    886346 non-null  int64  
 3   TARGET_FID    886346 non-null  int64  
 4   ID            886346 non-null  int64  
 5   X             886346 non-null  float64
 6   Y             886346 non-null  float64
 7   X_LP          886346 non-null  float64
 8   Y_LP          886346 non-null  float64
 9   NAME          634989 non-null  object 
 10  NAMELSAD      886346 non-null  object 
 11  Shape_Length  886346 non-null  float64
 12  Name          138699 non-null  object 
 13  Type          138699 non-null  object 
dtypes: float64(5), int64(4), object(5)
memory usage: 94.7+ MB


In [10]:
#merge the parcel crosswalk with each df
twentythree = parcel_cross.merge(twentythree, on = 'ID', how = 'outer')
fifty = parcel_cross.merge(fifty, on = 'ID', how = 'outer')

In [11]:
fifty.tail(2)

,OBJECTID *,Shape *,Join_Count,TARGET_FID,ID,X,Y,X_LP,Y_LP,NAME,NAMELSAD,Shape_Length,Name,Type,year,geo_level,x,y,units,households,jobs,household_population
886344,553138,Point,1,553137,891623,-86.504976,36.094515,-86.504976,36.094515,NaN,Wilson,1.961453e+06,NaN,NaN,2050,parcel,-86.504976,36.094515,1,1,0.0,2.37
886345,553140,Point,1,553139,891575,-86.509886,36.097989,-86.509886,36.097989,NaN,Wilson,1.961453e+06,NaN,NaN,2050,parcel,-86.509886,36.097989,2,2,0.0,4.73


In [12]:
twentythree.tail(2)

,OBJECTID *,Shape *,Join_Count,TARGET_FID,ID,X,Y,X_LP,Y_LP,NAME,NAMELSAD,Shape_Length,Name,Type,year,geo_level,x,y,units,households,jobs,household_population
886344,553138,Point,1,553137,891623,-86.504976,36.094515,-86.504976,36.094515,NaN,Wilson,1.961453e+06,NaN,NaN,2023,parcel,-86.504976,36.094515,1,1,0.0,2.35
886345,553140,Point,1,553139,891575,-86.509886,36.097989,-86.509886,36.097989,NaN,Wilson,1.961453e+06,NaN,NaN,2023,parcel,-86.509886,36.097989,2,2,0.0,4.71


Okay I need to consolidate all of the places, and unincorporated counties.

In [13]:
twentythree['NAME'].unique()

array([nan, 'Ashland City', 'Kingston Springs', 'Pegram', 'Pleasant View',
       'Belle Meade', 'Berry Hill', 'Forest Hills', 'Goodlettsville',
       'Metropolitan Nashville-Davidson County', 'Oak Hill', 'Ridgetop',
       'Burns', 'Charlotte', 'City of Dickson', 'Slayden', 'Vanleer',
       'White Bluff', 'Erin', 'Tennessee Ridge', 'McEwen',
       'New Johnsonville', 'Waverly', 'Columbia', 'Mount Pleasant',
       'Spring Hill', 'Clarksville', 'Adams', 'Cedar Hill', 'Coopertown',
       'Cross Plains', 'Greenbrier', 'Millersville', 'Orlinda',
       'Portland', 'Springfield', 'White House', 'Eagleville', 'LaVergne',
       'Murfreesboro', 'Smyrna', 'Cumberland City', 'Dover', 'Gallatin',
       'Hendersonville', 'Mitchellville', 'Westmoreland',
       'Trousdale/Hartsville', 'Brentwood', 'Fairview', 'Franklin',
       'Nolensville', "Thompson's Station", 'Lebanon', 'Mt. Juliet',
       'Watertown'], dtype=object)

In [14]:
twentythree = twentythree[['NAME', 'NAMELSAD', 'households', 'jobs', 'household_population']]
fifty = fifty[['NAME', 'NAMELSAD', 'households', 'jobs', 'household_population']]

In [15]:
coldict = {'NAME': 'Place', 
           'NAMELSAD': 'County', 
           'households': 'HHs', 
           'jobs': 'EMP', 
           'household_population': 'Pop'}
twentythree = twentythree.rename(columns = coldict)
fifty = fifty.rename(columns = coldict)

In [16]:
twentythree.head()

,Place,County,HHs,EMP,Pop
0,NaN,Cheatham,0,0.04,0.00
1,NaN,Cheatham,2,0.00,4.59
2,NaN,Cheatham,0,0.09,0.00
3,NaN,Cheatham,0,0.00,0.00
4,NaN,Cheatham,0,0.00,0.00


In [17]:
fifty.head()

,Place,County,HHs,EMP,Pop
0,NaN,Cheatham,0,0.04,0.0
1,NaN,Cheatham,0,0.00,0.0
2,NaN,Cheatham,0,0.09,0.0
3,NaN,Cheatham,0,0.00,0.0
4,NaN,Cheatham,0,0.00,0.0


In [18]:
#all the same places so just one list
places = list(twentythree['Place'].unique())

In [19]:
aggregated_places_23 = twentythree.groupby('Place', as_index=False)[['HHs', 'EMP', 'Pop']].sum()
aggregated_places_50 = fifty.groupby('Place', as_index=False)[['HHs', 'EMP', 'Pop']].sum()

In [20]:
aggregated_places_23.head(2)

,Place,HHs,EMP,Pop
0,Adams,238,112.85,613.31
1,Ashland City,2306,8728.31,5101.17


In [21]:
aggregated_places_50.head(2)

,Place,HHs,EMP,Pop
0,Adams,198,142.98,502.30
1,Ashland City,1947,10073.07,4208.98


In [22]:
#full name... there's both in Max's original so sure
aggregated_places_23['Place_Full'] = aggregated_places_23['Place'].map(place_to_full_name)
aggregated_places_50['Place_Full'] = aggregated_places_50['Place'].map(place_to_full_name)

In [23]:
aggregated_places_23.head(2)

,Place,HHs,EMP,Pop,Place_Full
0,Adams,238,112.85,613.31,"Adams city, Tennessee"
1,Ashland City,2306,8728.31,5101.17,"Ashland City town, Tennessee"


In [25]:
aggregated_places_50.head(2)

,Place,HHs,EMP,Pop,Place_Full
0,Adams,198,142.98,502.30,"Adams city, Tennessee"
1,Ashland City,1947,10073.07,4208.98,"Ashland City town, Tennessee"


In [26]:
aggregated_places_23 = aggregated_places_23.rename(columns = {'Place': 'Geography Name', 'Place_Full': 'Geography Full Name'})
aggregated_places_23['Geography'] = 'Census Place'
aggregated_places_50 = aggregated_places_50.rename(columns = {'Place': 'Geography Name', 'Place_Full': 'Geography Full Name'})
aggregated_places_50['Geography'] = 'Census Place'

In [27]:
aggregated_counties_23 = twentythree.groupby('County', as_index=False)[['HHs', 'EMP', 'Pop']].sum()
aggregated_counties_50 = fifty.groupby('County', as_index=False)[['HHs', 'EMP', 'Pop']].sum()

In [28]:
aggregated_counties_23.head(2)

,County,HHs,EMP,Pop
0,Cheatham,16100,17289.96,39841.46
1,Davidson,327705,753001.60,720771.08


In [29]:
aggregated_counties_50.head(2)

,County,HHs,EMP,Pop
0,Cheatham,18901,21235.53,47653.9
1,Davidson,360153,1047767.36,801751.7


In [30]:
uninc_23 = twentythree.loc[twentythree['Place'].isna()]
uninc_50 = fifty.loc[fifty['Place'].isna()]

In [31]:
uninc_23.head(2)

,Place,County,HHs,EMP,Pop
0,NaN,Cheatham,0,0.04,0.00
1,NaN,Cheatham,2,0.00,4.59


In [32]:
uninc_50.head(2)

,Place,County,HHs,EMP,Pop
0,NaN,Cheatham,0,0.04,0.0
1,NaN,Cheatham,0,0.00,0.0


In [33]:
aggregated_uninc_23 = uninc_23.groupby('County', as_index=False)[['HHs', 'EMP', 'Pop']].sum()
aggregated_uninc_50 = uninc_50.groupby('County', as_index=False)[['HHs', 'EMP', 'Pop']].sum()

In [34]:
aggregated_uninc_23['County'] = aggregated_uninc_23['County'] + " Unincorporated"
aggregated_uninc_50['County'] = aggregated_uninc_50['County'] + " Unincorporated"

In [35]:
aggregated_uninc_23 = aggregated_uninc_23.rename(columns = {'County': 'Geography Name'})
aggregated_uninc_23['Geography Full Name'] = aggregated_uninc_23['Geography Name']
aggregated_uninc_23['Geography'] = 'Unincorporated County'

In [36]:
aggregated_uninc_50 = aggregated_uninc_50.rename(columns = {'County': 'Geography Name'})
aggregated_uninc_50['Geography Full Name'] = aggregated_uninc_50['Geography Name']
aggregated_uninc_50['Geography'] = 'Unincorporated County'

In [37]:
aggregated_uninc_23

,Geography Name,HHs,EMP,Pop,Geography Full Name,Geography
0,Cheatham Unincorporated,9974,4268.00,25220.32,Cheatham Unincorporated,Unincorporated County
1,Dickson Unincorporated,11605,4588.35,29116.74,Dickson Unincorporated,Unincorporated County
2,Houston Unincorporated,2112,896.12,5017.93,Houston Unincorporated,Unincorporated County
3,Humphreys Unincorporated,3860,3430.80,8963.60,Humphreys Unincorporated,Unincorporated County
4,Maury Unincorporated,14861,11988.95,37457.13,Maury Unincorporated,Unincorporated County
5,Montgomery Unincorporated,24387,13165.74,67431.28,Montgomery Unincorporated,Unincorporated County
6,Robertson Unincorporated,11656,4174.13,30379.49,Robertson Unincorporated,Unincorporated County
7,Rutherford Unincorporated,37227,14363.78,101933.23,Rutherford Unincorporated,Unincorporated County
8,Stewart Unincorporated,4445,1668.19,10426.24,Stewart Unincorporated,Unincorporated County
9,Sumner Unincorporated,21838,4446.60,57256.11,Sumner Unincorporated,Unincorporated County


In [38]:
aggregated_places_23.head()

,Geography Name,HHs,EMP,Pop,Geography Full Name,Geography
0,Adams,238,112.85,613.31,"Adams city, Tennessee",Census Place
1,Ashland City,2306,8728.31,5101.17,"Ashland City town, Tennessee",Census Place
2,Belle Meade,1126,1032.03,2932.05,"Belle Meade city, Tennessee",Census Place
3,Berry Hill,1442,18236.42,2092.46,"Berry Hill city, Tennessee",Census Place
4,Brentwood,15428,77664.47,45510.67,"Brentwood city, Tennessee",Census Place


In [39]:
dfs = [aggregated_places_23, aggregated_uninc_23]
fulltwentythree = pd.concat(dfs)

In [40]:
fulltwentythree.tail()

,Geography Name,HHs,EMP,Pop,Geography Full Name,Geography
7,Rutherford Unincorporated,37227,14363.78,101933.23,Rutherford Unincorporated,Unincorporated County
8,Stewart Unincorporated,4445,1668.19,10426.24,Stewart Unincorporated,Unincorporated County
9,Sumner Unincorporated,21838,4446.60,57256.11,Sumner Unincorporated,Unincorporated County
10,Williamson Unincorporated,21568,10682.60,59277.93,Williamson Unincorporated,Unincorporated County
11,Wilson Unincorporated,26844,9595.71,69188.34,Wilson Unincorporated,Unincorporated County


In [41]:
dfs = [aggregated_places_50, aggregated_uninc_50]
fullfifty = pd.concat(dfs)

In [42]:
fullfifty.tail()

,Geography Name,HHs,EMP,Pop,Geography Full Name,Geography
7,Rutherford Unincorporated,64218,23546.20,179775.84,Rutherford Unincorporated,Unincorporated County
8,Stewart Unincorporated,5168,2198.49,12749.95,Stewart Unincorporated,Unincorporated County
9,Sumner Unincorporated,25405,6299.97,67649.32,Sumner Unincorporated,Unincorporated County
10,Williamson Unincorporated,43614,58211.23,121459.62,Williamson Unincorporated,Unincorporated County
11,Wilson Unincorporated,32672,15650.89,85167.22,Wilson Unincorporated,Unincorporated County


In [43]:
fulltwentythree = fulltwentythree.rename(columns = {'HHs': 'HHs 2023 Base', 
                                                    'EMP': 'EMP 2023 Base', 
                                                    'Pop': 'Pop 2023 Base'})
fullfifty = fullfifty.rename(columns = {'HHs': 'HHs 2050 Forecast', 
                                        'EMP': 'EMP 2050 Forecast', 
                                        'Pop': 'Pop 2050 Forecast'})

In [44]:
data = fulltwentythree.merge(fullfifty, on = ['Geography Name', 'Geography Full Name', 'Geography'], how = 'outer')

In [45]:
#set the order we want
data = data[['Geography Name', 'Geography Full Name', 'Geography', 'Pop 2023 Base', 'HHs 2023 Base', 'EMP 2023 Base', 
             'Pop 2050 Forecast', 'HHs 2050 Forecast', 'EMP 2050 Forecast']]

In [46]:
geos = list(data['Geography Full Name'].unique())

In [52]:
#bring in the 2020 PL
conn = sq.connect('../../Pipeline-Census-Bureau/Outputs/CensusBureau.db')
sql_query = pd.read_sql('SELECT * FROM [PL_2020]', conn)
pl2020 = pd.DataFrame(sql_query)
pl2020.drop(columns = 'Source', inplace = True)

In [53]:
pl2020 = pl2020.loc[pl2020['NAME'].isin(geos)]

In [54]:
pl2020 = pl2020[['NAME', 'Population', 'Occupancy:Occupied Units']]

In [55]:
pl2020 = pl2020.rename(columns = {'NAME': 'Geography Full Name', 
                                  'Population': 'Pop 2020 Decennial', 
                                  'Occupancy:Occupied Units': 'HHs 2020 Decennial'})

In [56]:
pl2020.tail(2)

,Geography Full Name,Pop 2020 Decennial,HHs 2020 Decennial
314,Williamson Unincorporated,54871.0,19060.0
316,Wilson Unincorporated,68464.0,25332.0


In [57]:
data = data.merge(pl2020, on = 'Geography Full Name', how = 'outer')

In [58]:
data.head(2)

,Geography Name,Geography Full Name,Geography,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast,Pop 2020 Decennial,HHs 2020 Decennial
0,Adams,"Adams city, Tennessee",Census Place,613.31,238,112.85,502.30,198,142.98,624.0,232.0
1,Ashland City,"Ashland City town, Tennessee",Census Place,5101.17,2306,8728.31,4208.98,1947,10073.07,5193.0,2145.0


In [59]:
#set order
data = data[['Geography Name', 'Geography Full Name', 'Geography', 'Pop 2020 Decennial', 'HHs 2020 Decennial', 
             'Pop 2023 Base', 'HHs 2023 Base', 'EMP 2023 Base', 
             'Pop 2050 Forecast', 'HHs 2050 Forecast', 'EMP 2050 Forecast']]

In [61]:
#bring in 2023 PEP
conn = sq.connect('../../Pipeline-Census-Bureau/Outputs/CensusBureau.db')
sql_query = pd.read_sql('SELECT * FROM [PEP_Place_2023Vintage]', conn)
pep = pd.DataFrame(sql_query)

In [62]:
pep.head()

,NAME,NAME_Full,2020 Population,2021 Population,2022 Population,2023 Population
0,Tennessee,"Tennessee, Tennessee",6926091,6963709,7048976,7126489
1,Adams city,"Adams city, Tennessee",621,631,646,653
2,Adamsville town,"Adamsville town, Tennessee",2262,2255,2250,2276
3,Alamo town,"Alamo town, Tennessee",2332,2333,2287,2333
4,Alcoa city,"Alcoa city, Tennessee",10968,11501,11684,13349


In [63]:
pep = pep.loc[pep['NAME_Full'].isin(geos)]

In [64]:
pep = pep.rename(columns = {'NAME_Full': 'Geography Full Name', '2023 Population': 'Pop 2023 PEP'})
pep = pep[['Geography Full Name', 'Pop 2023 PEP']]

In [65]:
data = data.merge(pep, on = 'Geography Full Name', how = 'outer')

In [66]:
#set order
data = data[['Geography Name', 'Geography Full Name', 'Geography', 'Pop 2020 Decennial', 'HHs 2020 Decennial', 'Pop 2023 PEP', 
             'Pop 2023 Base', 'HHs 2023 Base', 'EMP 2023 Base', 'Pop 2050 Forecast', 'HHs 2050 Forecast', 'EMP 2050 Forecast']]

In [71]:
data.head(2)

,Geography Name,Geography Full Name,Geography,Pop 2020 Decennial,HHs 2020 Decennial,Pop 2023 PEP,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast
0,Adams,"Adams city, Tennessee",Census Place,624.0,232.0,653.0,613.31,238,112.85,502.30,198,142.98
2,Ashland City,"Ashland City town, Tennessee",Census Place,5193.0,2145.0,5586.0,5101.17,2306,8728.31,4208.98,1947,10073.07


In [70]:
#don't know why there are duplicates but whatever they won't drop unless they're exact duplicates so not losing anything
data = data.drop_duplicates()

In [72]:
data.head(2)

,Geography Name,Geography Full Name,Geography,Pop 2020 Decennial,HHs 2020 Decennial,Pop 2023 PEP,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast
0,Adams,"Adams city, Tennessee",Census Place,624.0,232.0,653.0,613.31,238,112.85,502.30,198,142.98
2,Ashland City,"Ashland City town, Tennessee",Census Place,5193.0,2145.0,5586.0,5101.17,2306,8728.31,4208.98,1947,10073.07


In [73]:
# #as a test, bring in JobsEQ employment... not using it's not 1 to 1 for the employment in the model
# emp = pd.read_csv('../data/urbansim/JobsEQ_2023EMP.csv')
# emp = emp.rename(columns = {'Region': 'Geography Full Name', 
#                             'Empl': 'EMP 2023 JobsEQ'})
# emp.info()

In [74]:
# emp = emp.set_index('Geography Full Name').transpose()

In [75]:
# emp.head()

In [76]:
# #cheatham
# thelist = [emp['Ashland City town, Tennessee'], emp['Kingston Springs town, Tennessee'], emp['Pegram town, Tennessee'], emp['Pleasant View city, Tennessee']]
# cheathaminc = sum(thelist)
# emp['Cheatham Unincorporated'] = emp['Cheatham County, Tennessee'] - cheathaminc
# #dickson
# thelist = [emp['Burns town, Tennessee'], emp['Charlotte town, Tennessee'], emp['Dickson city, Tennessee'], emp['Slayden town, Tennessee'], 
#           emp['Vanleer town, Tennessee'], emp['White Bluff town, Tennessee']]
# dicksoninc = sum(thelist)
# emp['Dickson Unincorporated'] = emp['Dickson County, Tennessee'] - dicksoninc
# #humphreys
# thelist = [emp['McEwen city, Tennessee'], emp['New Johnsonville city, Tennessee'], emp['Waverly city, Tennessee']]
# humphreysinc = sum(thelist)
# emp['Humphreys Unincorporated'] = emp['Humphreys County, Tennessee'] - humphreysinc
# #montgomery
# thelist = [emp['Clarksville city, Tennessee']]
# montgomeryinc = sum(thelist)
# emp['Montgomery Unincorporated'] = emp['Montgomery County, Tennessee'] - montgomeryinc
# #rutherford
# thelist = [emp['Eagleville city, Tennessee'], emp['La Vergne city, Tennessee'], emp['Murfreesboro city, Tennessee'], emp['Dover city, Tennessee']]
# rutherfordinc = sum(thelist)
# emp['Rutherford Unincorporated'] = emp['Rutherford County, Tennessee'] - rutherfordinc
# #wilson
# thelist = [emp['Lebanon city, Tennessee'], emp['Mount Juliet city, Tennessee'], emp['Watertown city, Tennessee']]
# wilsoninc = sum(thelist)
# emp['Wilson Unincorporated'] = emp['Wilson County, Tennessee'] - wilsoninc

In [77]:
# emp = emp.transpose().reset_index(drop = False)
# emp.head(2)

In [78]:
# data = data.merge(emp, on = 'Geography Full Name', how = 'left')

In [79]:
data = data[['Geography Name', 'Geography Full Name', 'Geography', 'Pop 2020 Decennial', 'HHs 2020 Decennial', 'Pop 2023 PEP', #'EMP 2023 JobsEQ',  
             'Pop 2023 Base', 'HHs 2023 Base', 'EMP 2023 Base', 'Pop 2050 Forecast', 'HHs 2050 Forecast', 'EMP 2050 Forecast']]

In [80]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69 entries, 0 to 120
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Geography Name       69 non-null     object 
 1   Geography Full Name  69 non-null     object 
 2   Geography            69 non-null     object 
 3   Pop 2020 Decennial   68 non-null     float64
 4   HHs 2020 Decennial   68 non-null     float64
 5   Pop 2023 PEP         57 non-null     float64
 6   Pop 2023 Base        69 non-null     float64
 7   HHs 2023 Base        69 non-null     int64  
 8   EMP 2023 Base        69 non-null     float64
 9   Pop 2050 Forecast    69 non-null     float64
 10  HHs 2050 Forecast    69 non-null     int64  
 11  EMP 2050 Forecast    69 non-null     float64
dtypes: float64(7), int64(2), object(3)
memory usage: 7.0+ KB


In [81]:
cols = list(data.columns)
cols.remove('Geography Name')
cols.remove('Geography Full Name')
cols.remove('Geography')


In [82]:
cols

['Pop 2020 Decennial',
 'HHs 2020 Decennial',
 'Pop 2023 PEP',
 'Pop 2023 Base',
 'HHs 2023 Base',
 'EMP 2023 Base',
 'Pop 2050 Forecast',
 'HHs 2050 Forecast',
 'EMP 2050 Forecast']

In [182]:
data[cols] = round(data[cols], 0)
data[cols] = data[cols].apply(lambda x: round(x).astype("Int64"))

In [83]:
data['Abs Variation'] = data['Pop 2023 Base'] - data['Pop 2023 PEP']
data['Pct Variation'] = data['Abs Variation']/data['Pop 2023 PEP']
#data['Abs Variation EMP'] = data['EMP 2023 Base'] - data['EMP 2023 JobsEQ']
#data['Pct Variation EMP'] = data['Abs Variation EMP']/data['EMP 2023 JobsEQ']
data['Pop Net Change 2023-2050'] = (data['Pop 2050 Forecast'] - data['Pop 2023 Base'])/data['Pop 2023 Base']
data['HHs Net Change 2023-2050'] = (data['HHs 2050 Forecast'] - data['HHs 2023 Base'])/data['HHs 2023 Base']
data['EMP Net Change 2023-2050'] = (data['EMP 2050 Forecast'] - data['EMP 2023 Base'])/data['EMP 2023 Base']

In [84]:
data.head()

,Geography Name,Geography Full Name,Geography,Pop 2020 Decennial,HHs 2020 Decennial,Pop 2023 PEP,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast,Abs Variation,Pct Variation,Pop Net Change 2023-2050,HHs Net Change 2023-2050,EMP Net Change 2023-2050
0,Adams,"Adams city, Tennessee",Census Place,624.0,232.0,653.0,613.31,238,112.85,502.30,198,142.98,-39.69,-0.060781,-0.181001,-0.168067,0.266992
2,Ashland City,"Ashland City town, Tennessee",Census Place,5193.0,2145.0,5586.0,5101.17,2306,8728.31,4208.98,1947,10073.07,-484.83,-0.086794,-0.174899,-0.155681,0.154069
4,Belle Meade,"Belle Meade city, Tennessee",Census Place,2901.0,1073.0,2594.0,2932.05,1126,1032.03,3279.49,1255,1005.03,338.05,0.130320,0.118497,0.114565,-0.026162
7,Berry Hill,"Berry Hill city, Tennessee",Census Place,2112.0,1446.0,1840.0,2092.46,1442,18236.42,2177.93,1451,21975.92,252.46,0.137207,0.040847,0.006241,0.205057
10,Brentwood,"Brentwood city, Tennessee",Census Place,45373.0,14741.0,45265.0,45510.67,15428,77664.47,62517.60,20922,120058.99,245.67,0.005427,0.373691,0.356106,0.545868


In [85]:
data.tail()

,Geography Name,Geography Full Name,Geography,Pop 2020 Decennial,HHs 2020 Decennial,Pop 2023 PEP,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast,Abs Variation,Pct Variation,Pop Net Change 2023-2050,HHs Net Change 2023-2050,EMP Net Change 2023-2050
116,Rutherford Unincorporated,Rutherford Unincorporated,Unincorporated County,96115.0,33368.0,NaN,101933.23,37227,14363.78,179775.84,64218,23546.20,NaN,NaN,0.763663,0.725038,0.639276
117,Stewart Unincorporated,Stewart Unincorporated,Unincorporated County,11526.0,4673.0,NaN,10426.24,4445,1668.19,12749.95,5168,2198.49,NaN,NaN,0.222871,0.162655,0.317889
118,Sumner Unincorporated,Sumner Unincorporated,Unincorporated County,56008.0,20412.0,NaN,57256.11,21838,4446.60,67649.32,25405,6299.97,NaN,NaN,0.181521,0.163339,0.416806
119,Williamson Unincorporated,Williamson Unincorporated,Unincorporated County,54871.0,19060.0,NaN,59277.93,21568,10682.60,121459.62,43614,58211.23,NaN,NaN,1.048986,1.022162,4.449163
120,Wilson Unincorporated,Wilson Unincorporated,Unincorporated County,68464.0,25332.0,NaN,69188.34,26844,9595.71,85167.22,32672,15650.89,NaN,NaN,0.230948,0.217106,0.631030


In [87]:
data = data[['Geography Name', 'Geography Full Name', 'Geography', 'Pop 2020 Decennial', 'HHs 2020 Decennial', 'Pop 2023 PEP', #'EMP 2023 JobsEQ', 
             'Pop 2023 Base', 'HHs 2023 Base', 'EMP 2023 Base', 'Abs Variation', 'Pct Variation', #'Abs Variation EMP', 'Pct Variation EMP', 
             'Pop 2050 Forecast', 'HHs 2050 Forecast', 'EMP 2050 Forecast', 'Pop Net Change 2023-2050', 'HHs Net Change 2023-2050', 'EMP Net Change 2023-2050']]
data = data.reset_index(drop = True)

In [88]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Geography Name            69 non-null     object 
 1   Geography Full Name       69 non-null     object 
 2   Geography                 69 non-null     object 
 3   Pop 2020 Decennial        68 non-null     float64
 4   HHs 2020 Decennial        68 non-null     float64
 5   Pop 2023 PEP              57 non-null     float64
 6   Pop 2023 Base             69 non-null     float64
 7   HHs 2023 Base             69 non-null     int64  
 8   EMP 2023 Base             69 non-null     float64
 9   Abs Variation             57 non-null     float64
 10  Pct Variation             57 non-null     float64
 11  Pop 2050 Forecast         69 non-null     float64
 12  HHs 2050 Forecast         69 non-null     int64  
 13  EMP 2050 Forecast         69 non-null     float64
 14  Pop Net Chan

In [91]:
data.to_csv('../data/urbansim/smallareaoutput.csv', index = False)

In [92]:
data.tail()

,Geography Name,Geography Full Name,Geography,Pop 2020 Decennial,HHs 2020 Decennial,Pop 2023 PEP,Pop 2023 Base,HHs 2023 Base,EMP 2023 Base,Abs Variation,Pct Variation,Pop 2050 Forecast,HHs 2050 Forecast,EMP 2050 Forecast,Pop Net Change 2023-2050,HHs Net Change 2023-2050,EMP Net Change 2023-2050
64,Rutherford Unincorporated,Rutherford Unincorporated,Unincorporated County,96115.0,33368.0,NaN,101933.23,37227,14363.78,NaN,NaN,179775.84,64218,23546.20,0.763663,0.725038,0.639276
65,Stewart Unincorporated,Stewart Unincorporated,Unincorporated County,11526.0,4673.0,NaN,10426.24,4445,1668.19,NaN,NaN,12749.95,5168,2198.49,0.222871,0.162655,0.317889
66,Sumner Unincorporated,Sumner Unincorporated,Unincorporated County,56008.0,20412.0,NaN,57256.11,21838,4446.60,NaN,NaN,67649.32,25405,6299.97,0.181521,0.163339,0.416806
67,Williamson Unincorporated,Williamson Unincorporated,Unincorporated County,54871.0,19060.0,NaN,59277.93,21568,10682.60,NaN,NaN,121459.62,43614,58211.23,1.048986,1.022162,4.449163
68,Wilson Unincorporated,Wilson Unincorporated,Unincorporated County,68464.0,25332.0,NaN,69188.34,26844,9595.71,NaN,NaN,85167.22,32672,15650.89,0.230948,0.217106,0.631030
